# Session 10 — Train-Test Split

**Goal:** build the actual disease classifier the whole module has been working toward,
and — the substantive change from Session 9 — evaluate it on patients it has never
seen, using a procedure that resists the two ways honest-looking evaluations go wrong:
leakage and single-split luck.

## What this stage does for the system

Session 9 ended with an unsolved problem: every metric was computed on the data the
model was fitted to, and an unrestricted decision tree scored a perfect R² by memorising
297 patients. No diagnostic run on training data can tell learning from memorisation.

This session installs the fix and the two traps that defeat it.

**Trap 1 — leakage.** If any information from the held-out patients influenced the
model, the held-out score is contaminated, and the contamination usually enters through
a preprocessing step applied before the split rather than through anything obviously
wrong. Step 5 builds a model out of *pure random noise* that scores an AUC of 0.80 on
its own test set, purely through leakage.

**Trap 2 — one split is a lottery.** A single 80/20 split on 297 patients puts 60 in
the test set, and 60 patients is not enough to pin down a performance number. Step 4
shows the same model and the same data producing test AUCs from 0.81 to 0.95 depending
only on which patients landed where.

## The dataset

Every session in this module works on one registry: the UCI **Heart Disease**
dataset (Cleveland), fetched live from the UCI ML Repository with `ucimlrepo` so the
notebooks are runnable by anyone without a CSV sitting on their machine. It holds 303
patients with clinical measurements (`age`, `trestbps` resting blood pressure, `chol`
serum cholesterol, `thalach` max heart rate achieved, `oldpeak` ST depression),
categorical findings (`sex`, `cp` chest-pain type, `fbs` fasting blood sugar > 120,
`restecg`, `exang` exercise-induced angina, `slope`, `ca`, `thal`), and the outcome
`num` — angiographic disease severity 0-4, which this module binarises into
`target` (0 = no disease, 1 = disease present).

Deliberately one dataset throughout: switching datasets between topics would mean
re-learning the data every session instead of building cumulative familiarity with
one problem, the way a real analyst does.

## How to read this notebook

Every code cell is followed by a short **Observe / Infer** note: *Observe* points at
exactly what to look at in that cell's output, and *Infer* explains what conclusion to
draw from it — and what a different result would imply. Read them before running the
next cell; several of them flag things worth double-checking before you move on.

## Prerequisites

This session runs entirely locally — no account or credentials needed.

```bash
pip install ucimlrepo pandas numpy scipy scikit-learn statsmodels matplotlib seaborn
```

## Step 1 — Load the registry from the UCI repository

Fetching directly from the UCI ML Repository keeps this notebook runnable by anyone,
instead of depending on a CSV already sitting on your machine. The same nine lines
open every session in this module, so the 297 patients below are the identical 297
patients every other notebook analyses.

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

heart_disease = fetch_ucirepo(id=45)
df = pd.concat([heart_disease.data.features, heart_disease.data.targets], axis=1)

# `num` is severity 0-4; this module screens for disease presence, so binarise it.
df = df.dropna().reset_index(drop=True)
df["target"] = (df["num"] > 0).astype(int)
df = df.drop(columns="num")

print(f"{len(df)} patients, {len(df.columns)} columns")
print(f"disease prevalence: {df['target'].mean():.3f}")
df.head()

**Observe:** `297 patients, 14 columns` and `disease prevalence: 0.461`. The preview
shows `age`, `sex`, `cp`, `trestbps`, `chol`, `fbs`, `restecg`, `thalach`, `exang`,
`oldpeak`, `slope`, `ca`, `thal`, and the `target` column just derived.
**Infer:** 303 rows are fetched and 297 survive `dropna()` — six patients are missing
`ca` (number of major vessels seen on fluoroscopy) or `thal`. Dropping six rows out of
303 is defensible here and keeps every notebook in this module working on the identical
297 patients; on a larger fraction of missing values you would have to impute instead,
and *that* choice would itself need the distribution work of Session 3. If your row
count is not 297, you are on a different subset than every number quoted below.

## Step 2 — Assemble the classifier

The target is now `target` (disease presence), so the model is **logistic regression**:
same linear structure as Session 9, passed through a logistic function so its output is
a probability between 0 and 1 — exactly the quantity Session 1 spent a whole notebook
learning to read.

The metric is **AUC** (area under the ROC curve): the probability that a randomly chosen
diseased patient is scored higher than a randomly chosen non-diseased one. 0.5 is
chance, 1.0 is perfect. Unlike accuracy, it does not depend on choosing a threshold and
is not flattered by class imbalance.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, accuracy_score

features = [c for c in df.columns if c != "target"]
X = df[features].to_numpy()
y = df["target"].to_numpy()

def build_model():
    # A Pipeline keeps scaling and fitting welded together -- Step 5 explains why.
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))

print(f"{len(features)} inputs: {', '.join(features)}")
print(f"{len(X)} patients, {y.mean():.1%} with disease")

**Observe:** thirteen inputs, 297 patients, 46.1% with disease.
**Infer:** all thirteen go in, including `chol` and `fbs`, which Sessions 6-8 found
non-significant individually — a variable can contribute in combination even when its
pairwise relationship is nil (Session 9's `sex` coefficient made the reverse point), and
with only thirteen candidates there is no pressing reason to prune. Two shortcuts are
being taken deliberately: the categorical codes (`cp`, `thal`, `slope`) are fed as
numbers rather than one-hot encoded, which Session 8 showed is not strictly right, and
`restecg`'s sparse category is left as-is. Both cost a little accuracy and neither
affects the evaluation lesson, which is the subject here. The near-balanced outcome is
convenient — with a 5% prevalence, accuracy would be a useless metric and even AUC would
need care.

## Step 3 — The mistake: fit and evaluate on the same patients

Session 9's unsolved problem, restated on the real target — and with a second model to
show the size of the problem depends on how much the model *can* memorise.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

fitted_on_all = build_model().fit(X, y)
train_auc = roc_auc_score(y, fitted_on_all.predict_proba(X)[:, 1])
print(f"logistic regression, scored on its own training data: AUC = {train_auc:.4f}")
print(f"                                              accuracy = {accuracy_score(y, fitted_on_all.predict(X)):.4f}")

tree_on_all = DecisionTreeClassifier(random_state=0).fit(X, y)
print(f"\nunrestricted tree, scored on its own training data: "
      f"AUC = {roc_auc_score(y, tree_on_all.predict_proba(X)[:, 1]):.4f}")

**Observe:** the logistic regression scores `0.9245` and the unrestricted tree scores a
flawless `1.0000` on the data each was fitted to.
**Infer:** the tree's perfect score is the reductio — it has a leaf per patient and has
learned nothing — but the regression's `0.9245` is the number that would actually get
reported, and it is inflated too, just by an unknown and smaller amount. The difference
between the two is *capacity*: the regression has fourteen parameters, so it cannot
memorise 297 patients even if it tried, while the tree effectively has one per patient.
That is the reason low-capacity models are more forgiving of this mistake, and also why
"my model isn't complex enough to overfit" is a claim to verify rather than assume.
Neither number here is evidence about performance on patient 298.

## Step 4 — Hold out patients the model never sees

Split first, fit on the training portion only, score once on the held-out portion.
`stratify=y` keeps the disease rate identical in both halves, so the test set is not
accidentally easier or harder than the training set.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

model = build_model().fit(X_train, y_train)
train_score = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
test_score = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

print(f"train: {len(X_train)} patients ({y_train.mean():.1%} disease)")
print(f"test:  {len(X_test)} patients ({y_test.mean():.1%} disease)")
print(f"\ntrain AUC: {train_score:.4f}")
print(f"test  AUC: {test_score:.4f}")
print(f"gap:       {train_score - test_score:+.4f}")

**Observe:** 237 training and 60 test patients with matched disease rates, a train AUC
of `0.9121` — and a test AUC of `0.9498`, which is **higher**.
**Infer:** a test score above the training score looks like a bug and is not one. The
model cannot have learned the test patients; what happened is that this particular draw
of 60 put an unusually separable set of patients in the test half. That is Session 4
speaking again: 60 patients is a small sample, its AUC has a wide standard error, and
this single number carries no more authority than the single convenience-sample estimate
did. Reporting `0.9498` as "the model's performance" would be exactly the mistake
Session 4 warned about — precision mistaken for accuracy. The next cell repeats the
split to size the problem.

In [ ]:
scores = []
for seed in range(30):
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    m = build_model().fit(Xa, ya)
    scores.append(roc_auc_score(yb, m.predict_proba(Xb)[:, 1]))

scores = np.array(scores)
print(f"test AUC across 30 different random splits of the SAME data:")
print(f"  min  {scores.min():.3f}")
print(f"  max  {scores.max():.3f}")
print(f"  mean {scores.mean():.3f}")
print(f"  sd   {scores.std():.3f}")
print(f"\nspread: {scores.max() - scores.min():.3f} AUC, from the split alone")

**Observe:** the same model on the same 297 patients yields test AUCs anywhere from
`0.810` to `0.953` — a spread of `0.143` — with a standard deviation of `0.034`.
**Infer:** the split *is* the experiment. Two teams running identical code with
different `random_state` values would report performance differing by more than a tenth
of an AUC, and either could pick the seed that flattered them without writing a single
dishonest line. That is the practical case for never reporting a single split's number,
and for fixing `random_state` explicitly so results are at least reproducible. The fix
that follows — averaging over folds — is Step 6.

## Step 5 — Leakage: the trap that survives a correct split

Leakage is any path by which held-out information reaches the model. It does not
require a coding error, only a step performed *before* the split. To show it at full
strength, here is a dataset with no signal in it at all: 500 columns of pure random
noise.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

rng = np.random.default_rng(0)
noise = rng.normal(size=(len(df), 500))   # 500 columns of nothing

# WRONG: choose the best 10 columns using ALL patients, then split.
selector = SelectKBest(f_classif, k=10).fit(noise, y)
Xa, Xb, ya, yb = train_test_split(selector.transform(noise), y,
                                  test_size=0.3, random_state=0, stratify=y)
leaky = LogisticRegression(max_iter=2000).fit(Xa, ya)
leaky_auc = roc_auc_score(yb, leaky.predict_proba(Xb)[:, 1])

# RIGHT: split first, choose columns using the training patients only.
Xa2, Xb2, ya2, yb2 = train_test_split(noise, y, test_size=0.3, random_state=0, stratify=y)
selector2 = SelectKBest(f_classif, k=10).fit(Xa2, ya2)
honest = LogisticRegression(max_iter=2000).fit(selector2.transform(Xa2), ya2)
honest_auc = roc_auc_score(yb2, honest.predict_proba(selector2.transform(Xb2))[:, 1])

print("A model built from 500 columns of PURE RANDOM NOISE:")
print(f"  select features before splitting (leaky): test AUC = {leaky_auc:.3f}")
print(f"  select features after splitting (honest): test AUC = {honest_auc:.3f}")
print(f"  the truth:                                          0.500")

**Observe:** the leaky pipeline reports `0.800` on data containing no signal whatsoever,
while the honest pipeline reports `0.624`.
**Infer:** `0.800` from pure noise is what leakage buys, and note that nothing in the
leaky code is *wrong-looking* — it splits properly, fits only on training data, and
scores only on test data. The contamination happened one line earlier, when the feature
selector looked at all 297 outcomes to decide which columns to keep, thereby picking the
ones that happened to correlate with the test patients' outcomes too. The rule: **every
step that learns anything from the data — scaling, feature selection, imputation,
encoding, resampling — must be fitted inside the training fold only.** That is exactly
what `make_pipeline` in Step 2 enforces, and it is the reason to use a Pipeline rather
than transforming a frame by hand. Worth noting the honest number is `0.624`, not
`0.500`: a 90-patient test set has its own sampling noise, which is Session 4 setting
the floor on how precisely any of this can be measured.

## Step 6 — K-fold cross-validation

Rather than trusting one split, partition the patients into k folds, train on k−1 and
test on the held-out one, rotate, and average. Every patient is used for testing exactly
once, so the estimate uses all the data and the fold-to-fold spread is itself
informative.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = cross_val_score(build_model(), X, y, cv=cv, scoring="roc_auc")

print("5-fold cross-validated AUC:")
for i, s in enumerate(fold_scores, 1):
    print(f"  fold {i}: {s:.3f}")
print(f"\nmean: {fold_scores.mean():.3f}  sd: {fold_scores.std():.3f}")
print(f"\nfor comparison:")
print(f"  fitted and scored on all patients (Step 3): {train_auc:.3f}   <- optimistic")
print(f"  one 80/20 split, seed 42 (Step 4):          {test_score:.3f}   <- a lottery ticket")
print(f"  5-fold cross-validation:                    {fold_scores.mean():.3f}   <- report this")

**Observe:** fold scores of `0.939, 0.912, 0.834, 0.859, 0.925` averaging `0.894` with
sd `0.040` — against `0.925` from the everything-at-once fit and `0.950` from the lucky
single split.
**Infer:** `0.894` is the number to report, and the fold spread is the reason: individual
folds range across a tenth of an AUC, so any single-split figure is a draw from that
range dressed up as a measurement. Both alternatives above overstate performance, for
different reasons — Step 3's by evaluating on training data, Step 4's by chance — which
is why they can agree with each other and still both be wrong. The choice of k trades
bias against cost: k=5 or 10 are the standard defaults, and k=n (leave-one-out) uses the
most data per fit but is expensive and has high variance. `StratifiedKFold` preserves the
disease rate in every fold, which matters more the rarer the outcome.

## Step 7 — What the cross-validated estimate still does not cover

Cross-validation measures how well this procedure generalises to *unseen patients from
this registry*. Several gaps remain, and they are worth stating explicitly before the
number gets quoted anywhere.

In [ ]:
gaps = pd.DataFrame([
    ("Same population?", "CV resamples THIS registry; a referral clinic differs from a "
                         "general population", "Session 1's prevalence caveat"),
    ("Same era?",        "all patients are from one collection period; practice and "
                         "populations drift", "monitoring, not statistics"),
    ("Model selection?", "tuning choices made by looking at CV scores leak into the CV "
                         "estimate itself", "nested CV, or a final untouched hold-out"),
    ("Enough patients?", "60-patient folds give AUC a standard error you cannot argue "
                         "away", "Session 4"),
    ("Right metric?",    "AUC ignores calibration -- correct RANKING can still mean "
                         "wrong PROBABILITIES", "Session 1"),
], columns=["question", "why CV doesn't answer it", "what does"])
print(gaps.to_string(index=False))

**Observe:** five gaps, of which only the fourth is a statistical-power question that
more patients would close.
**Infer:** the third row is the one that bites in practice and is easy to commit
accidentally: every time you look at a cross-validation score and change something — an
input, a hyperparameter, a preprocessing step — you spend a little of the CV estimate's
independence, and after enough iterations it is a training score wearing a hold-out's
clothes. The defence is a final test set locked away and touched exactly once, or nested
cross-validation. The last row matters specifically for this system: AUC only measures
whether diseased patients are *ranked* above non-diseased ones, so a model can achieve
0.89 while its predicted probabilities are systematically too high — and Session 1
showed that a probability is precisely what a clinician needs to act on. Ranking and
calibration are different properties, and this module has only measured one.

## What this session hands to the next one

- **An honestly evaluated classifier**: 5-fold cross-validated AUC `0.894`, with the
  fold spread reported alongside it.
- **A pipeline discipline** — every fitted transformation inside the training fold —
  demonstrated against a noise dataset that scored 0.80 without it.
- **The knowledge that a single split is a lottery**, with the spread measured.
- **An honest measurement of the train-test gap**, which is the diagnostic Session 11
  turns into a tool: if the gap is what reveals overfitting, then watching it grow as a
  model gets more complex shows exactly where "more complex" starts hurting.

## Try it yourself

1. Change `test_size` to 0.4 and re-run Step 4's 30-seed loop. Does a larger test set
   narrow the spread, and what does it cost the training set?
2. Replace `StratifiedKFold` with plain `KFold` in Step 6. How much do the fold scores
   move, and would the difference be larger or smaller with a rarer outcome?
3. Drop `chol` and `fbs` (non-significant in Sessions 6-8) and re-run Step 6. Does
   removing them help, hurt, or make no difference?
4. Repeat Step 5's leakage demo with the real inputs instead of noise. Is the leaky
   advantage still visible, and why is it so much smaller?